# Bagging (Bootstrap Aggregating) From Scratch

## What is Bagging?

A single model trained on one dataset can be unstable — small changes in the training data
can produce very different predictions. This is called **high variance**.

Bagging fixes this by training **many copies of the same model** on slightly different
versions of the data, then combining their predictions.

Think of it like asking 10 doctors for a diagnosis instead of just one.
Each doctor has seen slightly different patients. The majority opinion is more reliable
than any single doctor.

## The two key ideas

### 1. Bootstrap sampling
To create slightly different datasets, we sample **with replacement** from the original data.
This means the same row can appear more than once in a sample.

From 10 rows, a bootstrap sample of size 10 will typically contain:
- ~6-7 unique rows (some repeated)
- ~3-4 rows never selected (called **out-of-bag** rows)

Each bootstrap sample is slightly different, so each model trained on it will be slightly different.

### 2. Aggregation
- **Classification** — take the majority vote across all models
- **Regression** — take the average prediction across all models

## What bagging reduces

| Problem | Effect |
|---------|--------|
| **Variance** | Reduced significantly — averaging smooths out individual model quirks |
| **Bias** | Unchanged — each model is the same type, same complexity |

Bagging works best on **high-variance, low-bias** models like deep decision trees.
Random Forest is bagging applied to decision trees with one extra trick (feature randomness).

## Step 1 — The Dataset

We use a noisy binary classification problem with 200 samples.
The classes overlap slightly so no single model will be perfect — this is where bagging helps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 200

# Two overlapping Gaussian clusters
X0 = np.random.randn(n // 2, 2) + np.array([0, 0])
X1 = np.random.randn(n // 2, 2) + np.array([2, 2])
X  = np.vstack([X0, X1])
y  = np.array([0]*(n//2) + [1]*(n//2))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

plt.figure(figsize=(6, 5))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], alpha=0.6, label='Class 0')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], alpha=0.6, label='Class 1')
plt.title('Dataset (overlapping classes — not perfectly separable)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 2 — Bootstrap Sampling

The core of bagging is creating different training sets from the same data
by sampling **with replacement**.

From N training rows, we sample N rows — but since we sample with replacement,
some rows appear more than once and some never appear.

Mathematically, each row has a `(1 - 1/N)^N ≈ 36.8%` chance of never being selected.
So about **63.2%** of rows are unique in each bootstrap sample.

In [ ]:
def bootstrap_sample(X, y):
    n = len(X)
    indices = np.random.choice(n, size=n, replace=True)  # sample WITH replacement
    return X[indices], y[indices], indices

# Demonstrate what a bootstrap sample looks like
np.random.seed(0)
n_demo = 10
X_demo = np.arange(n_demo).reshape(-1, 1)
y_demo = np.zeros(n_demo, dtype=int)

_, _, idx = bootstrap_sample(X_demo, y_demo)
all_indices = np.arange(n_demo)
oob_indices = np.setdiff1d(all_indices, idx)

print(f'Original rows:        {list(all_indices)}')
print(f'Bootstrap sample idx: {sorted(idx.tolist())}')
print(f'Selected (unique):    {sorted(set(idx.tolist()))}')
print(f'Out-of-bag (missed):  {oob_indices.tolist()}')
print()
print(f'Unique rows selected: {len(set(idx))} / {n_demo}  ({100*len(set(idx))/n_demo:.0f}%)')
print(f'Out-of-bag rows:      {len(oob_indices)} / {n_demo}  ({100*len(oob_indices)/n_demo:.0f}%)')

## Step 3 — Why Does a Single Deep Tree Have High Variance?

Before building the bagging ensemble, we show the problem it solves.

A deep decision tree fits the training data very closely — it has low training error
but its test accuracy can vary a lot depending on which data it was trained on.
This is **high variance**.

We train 10 individual trees on different bootstrap samples and look at how different their
accuracies are.

In [ ]:
n_trees = 10
individual_accs = []

for i in range(n_trees):
    X_boot, y_boot, _ = bootstrap_sample(X_train, y_train)
    tree = DecisionTreeClassifier(max_depth=None, random_state=i)  # deep tree
    tree.fit(X_boot, y_boot)
    acc = accuracy_score(y_test, tree.predict(X_test))
    individual_accs.append(acc)
    print(f'  Tree {i+1:2d}  |  Test accuracy: {acc:.3f}')

print()
print(f'Mean accuracy:  {np.mean(individual_accs):.3f}')
print(f'Std deviation:  {np.std(individual_accs):.3f}  <- high variance')

## Step 4 — Bagging From Scratch

Now we implement the full bagging algorithm:

1. Create N bootstrap samples
2. Train one decision tree on each sample
3. For a new point, ask every tree to predict
4. Take the **majority vote** as the final prediction

The key insight is that each tree makes different mistakes
(because it was trained on different data).
When we take the majority vote, the mistakes tend to cancel out.

In [ ]:
class BaggingClassifier:
    def __init__(self, n_estimators=50, max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.random_state = random_state
        self.trees        = []

    def fit(self, X, y):
        self.trees = []
        np.random.seed(self.random_state)
        for i in range(self.n_estimators):
            X_boot, y_boot, _ = bootstrap_sample(X, y)
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=i)
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
        return self

    def predict(self, X):
        # Collect predictions from every tree: shape (n_estimators, n_samples)
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        # Majority vote for each sample
        return (all_preds.mean(axis=0) >= 0.5).astype(int)

# Train our bagging classifier
bag = BaggingClassifier(n_estimators=50, max_depth=None, random_state=42)
bag.fit(X_train, y_train)

bag_acc = accuracy_score(y_test, bag.predict(X_test))
print(f'Bagging (50 trees) test accuracy: {bag_acc:.3f}')

## Step 5 — How Accuracy Improves as We Add More Trees

We add trees one at a time and track the accuracy.
With just 1 tree, we get the noisy individual result.
As we add more trees, the majority vote stabilises and accuracy improves.

In [ ]:
# Collect all tree predictions upfront
np.random.seed(42)
max_trees = 100
all_preds = []

for i in range(max_trees):
    X_boot, y_boot, _ = bootstrap_sample(X_train, y_train)
    tree = DecisionTreeClassifier(max_depth=None, random_state=i)
    tree.fit(X_boot, y_boot)
    all_preds.append(tree.predict(X_test))

all_preds = np.array(all_preds)   # shape: (max_trees, n_test)

# Accuracy as we add trees one by one
accs = []
for k in range(1, max_trees + 1):
    vote = (all_preds[:k].mean(axis=0) >= 0.5).astype(int)
    accs.append(accuracy_score(y_test, vote))

# Single tree baseline
single_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
single_tree.fit(X_train, y_train)
single_acc = accuracy_score(y_test, single_tree.predict(X_test))

plt.figure(figsize=(8, 4))
plt.plot(range(1, max_trees + 1), accs, label='Bagging accuracy')
plt.axhline(single_acc, color='tomato', linestyle='--', label=f'Single tree ({single_acc:.3f})')
plt.title('Test Accuracy vs Number of Trees in Bagging')
plt.xlabel('Number of trees')
plt.ylabel('Test accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Single tree accuracy:      {single_acc:.3f}')
print(f'Bagging (100 trees):       {accs[-1]:.3f}')
print(f'Improvement:               +{accs[-1] - single_acc:.3f}')

## Step 6 — Visualise Decision Boundaries

A single deep tree creates a jagged, irregular boundary — it has memorised the training data.
The bagging ensemble creates a smoother boundary — it has learned the real pattern.

In [ ]:
def plot_boundary(model, X, y, ax, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', s=20, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='tomato',    s=20, label='Class 1')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_boundary(single_tree, X_test, y_test, axes[0],
              f'Single Deep Tree\n(Test acc = {single_acc:.3f})')
plot_boundary(bag, X_test, y_test, axes[1],
              f'Bagging (50 trees)\n(Test acc = {bag_acc:.3f})')
plt.suptitle('Decision Boundaries: Single Tree vs Bagging', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7 — Compare With sklearn's BaggingClassifier

In [ ]:
from sklearn.ensemble import BaggingClassifier as SklearnBagging

sk_bag = SklearnBagging(
    estimator=DecisionTreeClassifier(max_depth=None),
    n_estimators=50,
    random_state=42
)
sk_bag.fit(X_train, y_train)
sk_acc = accuracy_score(y_test, sk_bag.predict(X_test))

print(f'Our BaggingClassifier:     {bag_acc:.3f}')
print(f'sklearn BaggingClassifier: {sk_acc:.3f}')
print(f'Single tree baseline:      {single_acc:.3f}')

## Summary

| Concept | Explanation |
|---------|-------------|
| **Bootstrap sample** | Sample N rows with replacement from N rows — ~63% unique, ~37% out-of-bag |
| **High variance** | A single deep tree changes a lot depending on the training data it sees |
| **Aggregation** | Majority vote (classification) or average (regression) across all trees |
| **Why it works** | Each tree makes different errors — they cancel out in the vote |
| **What it reduces** | Variance — not bias |
| **Best base model** | High-variance, low-bias learners (deep decision trees) |

## Bagging vs Voting Ensemble (what the old notebook had)

| | Bagging | Voting Ensemble |
|-|---------|----------------|
| Base models | Same type (e.g. all decision trees) | Different types (LR + DT + KNN) |
| Training data | Different bootstrap samples | Same dataset |
| Goal | Reduce variance | Combine different strengths |
| Random Forest | Direct extension of bagging | Not related |

## Connection to Random Forest

Random Forest = Bagging + one extra trick:
at each split, only a **random subset of features** is considered.
This makes the trees even more different from each other, which reduces variance further.